In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as dataset

import numpy as np
import matplotlib.pyplot as plt

In [2]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [3]:
class EncoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, mask=None):
    x = self.ln1(x + self.mha(x, x, x, mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [4]:
class DecoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.ln3 = nn.LayerNorm(d_model)
    self.mha1 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=True)
    self.mha2 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):
    # self-attention on decoder input
    x = self.ln1(dec_input + self.mha1(dec_input, dec_input, dec_input, dec_mask))

    # multi-head attention including incoder output
    x = self.ln2(x + self.mha2(x, enc_output, enc_output, enc_mask))

    x = self.ln3(x + self.ann(x))
    x = self.dropout(x)
    return x

In [5]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=2048, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [6]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers,
               #n_classes,
               dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    #self.fc = nn.Linear(d_model, n_classes)

  def forward(self, x, mask=None):
    x = self.embedding(x)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    #x = x[:, 0, :]

    x = self.ln(x)
    #x = self.fc(x)
    return x

In [7]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers, dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    decoder_blocks = [
        DecoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.decoder_blocks = nn.Sequential(*decoder_blocks)
    self.ln = nn.LayerNorm(d_model)
    self.fc = nn.Linear(d_model, vocab_size)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):
    x = self.embedding(dec_input)
    x = self.pos_encoding(x)

    for block in self.decoder_blocks:
      x = block(enc_output, x, enc_mask, dec_mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    # x = x[:, 0, :] - this is for encoder

    x = self.ln(x)
    x = self.fc(x) # many-to-many
    return x

In [8]:
class Transformer(nn.Module):
  def __init__(self, encoder, decoder) -> None:
    super().__init__()

    self.encoder = encoder
    self.decoder = decoder

  def forward(self, enc_input, dec_input, enc_mask, dec_mask):
    enc_output = self.encoder(enc_input, enc_mask)
    dec_output = self.decoder(enc_output, dec_input, enc_mask, dec_mask)

    return dec_output

In [9]:
encoder = Encoder(vocab_size=20_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=10_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [10]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(10000, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [11]:
xe = np.random.randint(0, 20_000, size=(8, 512))
xe_t = torch.tensor(xe).to(device)

In [12]:
xd = np.random.randint(0, 10_000, size=(8, 256))
xd_t = torch.tensor(xd).to(device)

In [13]:
maske = np.ones((8, 512))
maske[:, 256:] = 0
maske_t = torch.tensor(maske).to(device)

maskd = np.ones((8, 256))
maskd[:, 128:] = 0
maskd_t = torch.tensor(maskd).to(device)

In [14]:
out = transformer(xe_t, xd_t, maske_t, maskd_t)
out.shape

torch.Size([8, 256, 10000])

In [15]:
out

tensor([[[ 0.2745,  0.3522, -0.8376,  ..., -0.7875, -0.1800,  0.3845],
         [ 1.0311,  0.2934,  0.2656,  ...,  0.0346,  0.1449, -0.2080],
         [-0.6463,  0.3254, -0.1504,  ..., -0.5584, -0.8044,  1.0416],
         ...,
         [-0.0503, -0.3766,  0.5950,  ...,  0.8482, -0.8282,  0.2873],
         [-0.3803,  0.5115, -0.7051,  ...,  0.0368, -0.9858,  0.0200],
         [ 0.6924, -0.1578,  0.5105,  ...,  0.1355, -0.0667,  0.2766]],

        [[-0.1151,  0.0859,  0.5069,  ...,  0.5732, -0.2661,  0.9606],
         [ 0.8842,  0.3591, -0.7644,  ...,  0.6047,  0.5906,  0.1634],
         [ 0.2305, -0.3640,  0.3440,  ..., -0.3543,  0.4602,  0.2478],
         ...,
         [ 0.0935,  0.1301,  0.1378,  ..., -0.1840, -0.5632, -0.3479],
         [ 0.6715, -1.3356, -0.3817,  ..., -1.2650, -0.7344, -0.4507],
         [-0.5843, -0.5120, -0.1591,  ..., -0.2210, -0.8670,  0.1638]],

        [[ 0.2515, -0.7232,  0.0915,  ...,  0.2168, -0.4433, -0.4245],
         [ 0.8094,  0.6611,  0.2357,  ...,  0

In [16]:
!head spa.txt

Go.	Ve.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986655 (cueyayotl)
Go.	Vete.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986656 (cueyayotl)
Go.	Vaya.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #4986657 (cueyayotl)
Go.	Váyase.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #6586271 (arh)
Go.	Id.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703671 (Seael)
Go.	Vayan.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703672 (Seael)
Go.	Váyanse.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #13703674 (Seael)
Hi.	Hola.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #431975 (Leono)
Run!	¡Corre!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1685404 (Elenitigormiti)
Run!	¡Corran!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #5213896 (cueyayotl)


In [17]:
import pandas as pd

df = pd.read_csv('spa.txt', sep="\t", header=None)
df.head()

,0,1,2
0,Go.,Ve.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Vete.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Vaya.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Go.,Váyase.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
4,Go.,Id.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...


In [18]:
df.shape

(144215, 3)

In [19]:
df = df.iloc[:30_000]

In [20]:
df.columns = ['en', 'es', 'd']

In [21]:
df

,en,es,d
0,Go.,Ve.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Vete.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Vaya.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Go.,Váyase.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
4,Go.,Id.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
...,...,...,...
29995,He studies very hard.,Él estudia muy duro.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
29996,He suddenly fell ill.,Él se enfermó de súbito.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
29997,He talked to himself.,Él habló consigo mismo.,CC-BY 2.0 (France) Attribution: tatoeba.org #3...
29998,He talks to me a lot.,Él me habla mucho.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...


In [22]:
df = df.drop('d', axis=1)

In [23]:
df

,en,es
0,Go.,Ve.
1,Go.,Vete.
2,Go.,Vaya.
3,Go.,Váyase.
4,Go.,Id.
...,...,...
29995,He studies very hard.,Él estudia muy duro.
29996,He suddenly fell ill.,Él se enfermó de súbito.
29997,He talked to himself.,Él habló consigo mismo.
29998,He talks to me a lot.,Él me habla mucho.


In [24]:
df.to_csv('spa.csv', index=None)

In [25]:
!head spa.csv

en,es
Go.,Ve.
Go.,Vete.
Go.,Vaya.
Go.,Váyase.
Go.,Id.
Go.,Vayan.
Go.,Váyanse.
Hi.,Hola.
Run!,¡Corre!


In [26]:
!pip install transformers datasets sentencepiece sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 23.7 MB/s eta 0:00:00


In [27]:
from datasets import load_dataset
raw_dataset = load_dataset('csv', data_files='spa.csv')

Generating train split: 0 examples [00:00, ? examples/s]

In [28]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'es'],
        num_rows: 30000
    })
})

In [29]:
split = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['en', 'es'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['en', 'es'],
        num_rows: 9000
    })
})

In [30]:
from transformers import AutoTokenizer

model_checkpoint = 'Helsinki-NLP/opus-mt-en-es'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

In [31]:
en_sentence = split['train'][0]['en']
es_sentence = split['train'][0]['es']

inputs = tokenizer(en_sentence)
targets = tokenizer(text_target=es_sentence)

tokenizer.convert_ids_to_tokens(targets['input_ids'])

['▁Esto', '▁lo', '▁sé', '.', '</s>']

In [32]:
es_sentence

'Esto lo sé.'

In [33]:
max_input_length = 128
max_target_length = 128

In [34]:
def preprocess_function(batch):
  model_inputs = tokenizer(batch['en'], max_length=max_input_length, truncation=True)
  labels = tokenizer(text_target=batch['es'], max_length=max_target_length, truncation=True)

  model_inputs['labels'] = labels['input_ids']
  return model_inputs

In [35]:
tokenized_datasets = split.map(preprocess_function, batched=True, remove_columns=split['train'].column_names)

Map:   0%|          | 0/21000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

In [36]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9000
    })
})

In [37]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer)

In [38]:
batch = data_collator([tokenized_datasets['train'][i] for i in range(0, 5)])
batch.keys()

KeysView({'input_ids': tensor([[   33,   161,    58,     3,     0, 65000, 65000],
        [   99,    20,   100, 14461,    74,     3,     0],
        [  160,  9656,     8, 14614,     3,     0, 65000],
        [   99,    20,   100,   125,  6723,     3,     0],
        [   33,   805,  3868,  6753,     3,     0, 65000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]]), 'labels': tensor([[  938,    57,   915,     3,     0,  -100,  -100],
        [  377,  1221,   129, 25866, 30684,     3,     0],
        [ 1782,  3688,    38, 12130,     3,     0,  -100],
        [ 4846,   155,  6818,     3,     0,  -100,  -100],
        [ 3732, 32251,  7573,     3,     0,  -100,  -100]])})

In [39]:
batch['input_ids']

tensor([[   33,   161,    58,     3,     0, 65000, 65000],
        [   99,    20,   100, 14461,    74,     3,     0],
        [  160,  9656,     8, 14614,     3,     0, 65000],
        [   99,    20,   100,   125,  6723,     3,     0],
        [   33,   805,  3868,  6753,     3,     0, 65000]])

In [40]:
batch['attention_mask']

tensor([[1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0]])

In [41]:
batch['labels']

tensor([[  938,    57,   915,     3,     0,  -100,  -100],
        [  377,  1221,   129, 25866, 30684,     3,     0],
        [ 1782,  3688,    38, 12130,     3,     0,  -100],
        [ 4846,   155,  6818,     3,     0,  -100,  -100],
        [ 3732, 32251,  7573,     3,     0,  -100,  -100]])

In [42]:
tokenizer.all_special_ids

[0, 1, 65000]

In [43]:
tokenizer.all_special_tokens

['</s>', '<unk>', '<pad>']

In [44]:
tokenizer('<pad>')

{'input_ids': [65000, 0], 'attention_mask': [1, 1]}

In [45]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

valid_loader = DataLoader(
    tokenized_datasets['test'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

In [46]:
for batch in train_loader:
  for k, v in batch.items():
    print("k: ", k, "v.shape: ", v.shape)
  break

k:  input_ids v.shape:  torch.Size([32, 9])
k:  attention_mask v.shape:  torch.Size([32, 9])
k:  labels v.shape:  torch.Size([32, 10])


In [47]:
tokenizer.vocab_size

65001

In [48]:
tokenizer.decode(60000)

'ѕэр'

In [49]:
tokenizer.add_special_tokens({"cls_token": "<s>"})

1

In [50]:
tokenizer('<s>')

{'input_ids': [65001, 0], 'attention_mask': [1, 1]}

In [51]:
tokenizer.decode(tokenizer('<s>')['input_ids'])

'<s> </s>'

In [52]:
tokenizer.vocab_size

65001

In [53]:
encoder = Encoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [54]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(65002, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [55]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(transformer.parameters())

In [56]:
from datetime import datetime

def train(model, criterion, optimizer, train_loader, valid_loader, epochs):
  train_losses = np.zeros(epochs)
  test_losses = np.zeros(epochs)

  for it in range(epochs):
    model.train()
    t0 = datetime.now()
    train_loss = []
    for batch in train_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      optimizer.zero_grad()

      enc_input = batch['input_ids']
      enc_mask = batch['attention_mask']
      targets = batch['labels']

      # shift targets forwards to get decoder_input
      dec_input = targets.clone().detach()
      dec_input = torch.roll(dec_input, shifts=1, dims=1)
      dec_input[:, 0] = 65_001

      # also convert all -100 to pad token id
      dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

      # make decoder input mask
      dec_mask = torch.ones_like(dec_input)
      dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


      # forward pass
      outputs = model(enc_input, dec_input, enc_mask, dec_mask)
      loss = criterion(outputs.transpose(2, 1), targets)

      # backward and optimizer
      loss.backward()
      optimizer.step()
      train_loss.append(loss.item())

    # get average train loss
    train_loss = np.mean(train_loss)

    model.eval()
    test_loss = []


    for batch in valid_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      # optimizer.zero_grad()

      enc_input = batch['input_ids']
      enc_mask = batch['attention_mask']
      targets = batch['labels']

      # shift targets forwards to get decoder_input
      dec_input = targets.clone().detach()
      dec_input = torch.roll(dec_input, shifts=1, dims=1)
      dec_input[:, 0] = 65_001

      # also convert all -100 to pad token id
      dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

      # make decoder input mask
      dec_mask = torch.ones_like(dec_input)
      dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


      # forward pass
      outputs = model(enc_input, dec_input, enc_mask, dec_mask)
      loss = criterion(outputs.transpose(2, 1), targets)

      # backward and optimizer
      # loss.backward()
      # optimizer.step()
      test_loss.append(loss.item())

    # get average train loss
    test_loss = np.mean(test_loss)



    train_losses[it] = train_loss
    test_losses[it] = test_loss

    dt = datetime.now() - t0
    print(f'Epoch {it+1}/{epochs}, Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, Duration: {dt}')

  return train_losses, test_losses

In [57]:
train_losses, test_losses = train(transformer, criterion, optimizer, train_loader, valid_loader, epochs=15)

Epoch 1/15, Train loss: 4.8874, Test loss: 3.8553, Duration: 0:00:17.719476
Epoch 2/15, Train loss: 3.5458, Test loss: 3.3075, Duration: 0:00:16.810217
Epoch 3/15, Train loss: 3.0563, Test loss: 2.9864, Duration: 0:00:20.385007
Epoch 4/15, Train loss: 2.6962, Test loss: 2.7885, Duration: 0:00:30.847517
Epoch 5/15, Train loss: 2.4100, Test loss: 2.6296, Duration: 0:00:25.603942
Epoch 6/15, Train loss: 2.1759, Test loss: 2.5148, Duration: 0:00:28.287360
Epoch 7/15, Train loss: 1.9855, Test loss: 2.4491, Duration: 0:00:26.318337
Epoch 8/15, Train loss: 1.8270, Test loss: 2.3902, Duration: 0:00:24.450298
Epoch 9/15, Train loss: 1.6958, Test loss: 2.3591, Duration: 0:00:22.571809
Epoch 10/15, Train loss: 1.5897, Test loss: 2.3439, Duration: 0:00:25.917575
Epoch 11/15, Train loss: 1.4973, Test loss: 2.3270, Duration: 0:00:18.828197
Epoch 12/15, Train loss: 1.4157, Test loss: 2.3139, Duration: 0:00:22.421040
Epoch 13/15, Train loss: 1.3544, Test loss: 2.3202, Duration: 0:00:22.544876
Epoch 14

In [58]:
input_sentence = split['test'][10]['en']
input_sentence

"I'll do it for you."

In [59]:
enc_input = tokenizer(input_sentence, return_tensors='pt')
enc_input

{'input_ids': tensor([[ 33,  20, 210, 113,  48,  23,  40,   3,   0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [60]:
dec_input_str = '<s>'

dec_input = tokenizer(text_target=dec_input_str, return_tensors='pt')
dec_input

{'input_ids': tensor([[65001,     0]]), 'attention_mask': tensor([[1, 1]])}

In [61]:
enc_input.to(device)
dec_input.to(device)
output = transformer(enc_input['input_ids'],
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

output

tensor([[[  2.6301, -14.6105,   3.7155,  ..., -13.6541, -13.6505, -13.9643]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [62]:
output.shape # N x T x V

torch.Size([1, 1, 65002])

In [63]:
enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])
enc_output.shape

torch.Size([1, 9, 64])

In [64]:
dec_output = decoder(enc_output,
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

dec_output.shape

torch.Size([1, 1, 65002])

In [66]:
torch.allclose(output, dec_output)

True

In [68]:
dec_input_ids = dec_input['input_ids'][:, :-1]
dec_attn_mask = dec_input['attention_mask'][:, :-1]

for _ in range(32):
  dec_output = decoder(enc_output,
                      dec_input_ids,
                      enc_input['attention_mask'],
                      dec_attn_mask)

  # choose the best value (or sample)
  prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

  # append to decoder input
  dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

  # recreate mask
  dec_attn_mask = torch.ones_like(dec_input_ids)

  # exit when reach </s>
  if prediction_id == 0:
    break


In [72]:
tokenizer.decode(dec_input_ids[0])

'<s> Te lo haré.</s>'

In [73]:
split['test'][10]['es']

'Lo haré para ti.'

In [76]:
def translate(input_sentence):
  # get encoder output first
  enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
  enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

  # setup initial decoder input
  dec_input_ids = torch.tensor([[65_001]], device=device)
  dec_attn_mask = torch.ones_like(dec_input_ids, device=device)

  # now do the decoder loop
  for _ in range(32):
    dec_output = decoder(enc_output,
                        dec_input_ids,
                        enc_input['attention_mask'],
                        dec_attn_mask)

    # choose the best value (or sample)
    prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

    # append to decoder input
    dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

    # recreate mask
    dec_attn_mask = torch.ones_like(dec_input_ids)

    # exit when reach </s>
    if prediction_id == 0:
      break

  translation = tokenizer.decode(dec_input_ids[0, 1:])
  print(translation)

In [77]:
translate("How are you?")

¿Cómo estás?</s>
